# Crude Oil Contracts - WTI vs Brent Split

Load `disaggregated_combined.csv`, filter to `commodity_name == 'CRUDE OIL'`,
then split the contracts by keyword in `market_and_exchange_names`:
- **WTI** - contains "WTI"
- **Brent** - contains "BRENT"
- **Other** - everything else (Dubai, Oman, WCS, generic names, etc.)

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../downloads/cftc/disaggregated_combined.csv", low_memory=False)
crude = df[df["commodity_name"] == "CRUDE OIL"].copy()
print(f"Total CRUDE OIL rows: {len(crude):,}")
print(f"Unique contract codes: {crude['cftc_contract_market_code'].nunique()}")

Total CRUDE OIL rows: 11,375
Unique contract codes: 36


In [3]:
# Get all unique (code, name) pairs
contracts = (
    crude[["cftc_contract_market_code", "market_and_exchange_names"]]
    .drop_duplicates()
    .sort_values(["cftc_contract_market_code", "market_and_exchange_names"])
    .reset_index(drop=True)
)

name = contracts["market_and_exchange_names"].str.upper()
has_wti = name.str.contains("WTI", regex=False)
has_brent = name.str.contains("BRENT", regex=False)

contracts["group"] = "Other"
contracts.loc[has_wti & ~has_brent, "group"] = "WTI"
contracts.loc[has_brent & ~has_wti, "group"] = "Brent"
contracts.loc[has_wti & has_brent, "group"] = "WTI-Brent Spread"

print(f"Total unique (code, name) pairs: {len(contracts)}")
print(f"\nBy group:")
print(contracts["group"].value_counts())

Total unique (code, name) pairs: 45

By group:
group
Other               17
WTI                 16
Brent                8
WTI-Brent Spread     4
Name: count, dtype: int64


## WTI Contracts

In [4]:
wti = contracts[contracts["group"] == "WTI"].sort_values("cftc_contract_market_code")
print(f"WTI rows: {len(wti)}  |  Unique codes: {wti['cftc_contract_market_code'].nunique()}")
wti

WTI rows: 16  |  Unique codes: 14


,cftc_contract_market_code,market_and_exchange_names,group
0,06739B,CRUDE DIFF-WCS CUSHING/WTI 1ST - ICE FUTURES E...,WTI
1,06739C,CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES E...,WTI
4,067411,"CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE",WTI
5,06741Q,WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV,WTI
8,06742R,ARGUS MARS vs WTI TRADE MONTH - ICE FUTURES EN...,WTI
10,06742U,ARGUS WTI Mid/WTI TRADE MONTH - ICE FUTURES EN...,WTI
12,06743D,ARGUS WTI HOUSTON/WTI TRADE MO - ICE FUTURES E...,WTI
14,067651,WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE,WTI
17,06765A,WTI CRUDE OIL CALENDAR - NEW YORK MERCANTILE E...,WTI
18,06765A,WTI CRUDE OIL CALENDAR SWAP - NEW YORK MERCANT...,WTI


## Brent Contracts

In [5]:
brent = contracts[contracts["group"] == "Brent"].sort_values("cftc_contract_market_code")
print(f"Brent rows: {len(brent)}  |  Unique codes: {brent['cftc_contract_market_code'].nunique()}")
brent

Brent rows: 8  |  Unique codes: 7


,cftc_contract_market_code,market_and_exchange_names,group
24,06765J,BRENT FINANCIAL - NEW YORK MERCANTILE EXCHANGE,Brent
27,06765M,DATED TO FRONTLINE BRENT SWAP - NEW YORK MERCA...,Brent
28,06765N,BRENT (ICE) CALENDAR SWAP - NEW YORK MERCANTIL...,Brent
29,06765O,BRENT-DUBAI SWAP - NEW YORK MERCANTILE EXCHANGE,Brent
30,06765T,BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE...,Brent
31,06765T,BRENT LAST DAY - NEW YORK MERCANTILE EXCHANGE,Brent
32,06765X,EUR STYLE BRENT OPTIONS - NEW YORK MERCANTILE ...,Brent
33,06765Y,BRENT AVG PRICE OPTIONS - NEW YORK MERCANTILE ...,Brent


## Other Contracts (Dubai, Oman, WCS, diffs, etc.)

In [ ]:
# Exclude codes that were already identified as WTI, Brent, or WTI-Brent Spread
known_codes = set(
    contracts[contracts["group"] != "Other"]["cftc_contract_market_code"]
)
other = contracts[
    (contracts["group"] == "Other")
    & (~contracts["cftc_contract_market_code"].isin(known_codes))
].sort_values("cftc_contract_market_code")
print(f"Other rows: {len(other)}  |  Unique codes: {other['cftc_contract_market_code'].nunique()}")
other

## Contracts that reference BOTH WTI and Brent

These are inter-commodity spreads (e.g. WTI-BRENT).

In [7]:
both = contracts[contracts["group"] == "WTI-Brent Spread"].sort_values("cftc_contract_market_code")
print(f"WTI-Brent Spread rows: {len(both)}  |  Unique codes: {both['cftc_contract_market_code'].nunique()}")
both

WTI-Brent Spread rows: 4  |  Unique codes: 3


,cftc_contract_market_code,market_and_exchange_names,group
6,06741R,WTI 1st Line-Brent 1st Line - ICE FUTURES ENER...,WTI-Brent Spread
25,06765L,WTI-BRENT CALENDAR - NEW YORK MERCANTILE EXCHANGE,WTI-Brent Spread
26,06765L,WTI-BRENT CALENDAR SWAP - NEW YORK MERCANTILE ...,WTI-Brent Spread
34,06765Z,WTI-BRENT SPREAD OPTION - NEW YORK MERCANTILE ...,WTI-Brent Spread
